# InsightVLM — NB3: Resolution Ablation @ 384 px

Charts encode meaning in thin lines, small markers and axis text. 224 px is the
ImageNet convention, not necessarily right here.

Same model and epoch budget as NB1 — **only the input resolution differs**, so
the comparison is clean. Batch is reduced to keep activations inside 16 GB.
~60 min. Output: `nb3_results.json`

In [ ]:
import os, sys, json, time, random, warnings, gc
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import torchvision.models as M

SEED = 20260815
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device", DEVICE,
      "|", torch.cuda.device_count(), "GPU(s)")

DATA_ROOT = Path("/kaggle/input/datasets/asifxzaman/xxxyyy/dataset/dataset")
def find_data():
    if (DATA_ROOT / "records.jsonl").exists(): return DATA_ROOT
    c = list(Path("/kaggle/input").glob("**/records.jsonl"))
    if not c: raise FileNotFoundError("records.jsonl not found under /kaggle/input")
    return c[0].parent
DATA = find_data(); print("dataset:", DATA)

meta = pd.read_parquet(DATA / "tables" / "metadata.parquet")
_ci = json.loads((DATA / "class_index.json").read_text())["label_to_id"]
CLASSES = sorted(_ci, key=_ci.get); NC = len(CLASSES); C2I = {c: i for i, c in enumerate(CLASSES)}
tr = meta[meta.split == "train"].reset_index(drop=True)
va = meta[meta.split == "val"].reset_index(drop=True)
te = meta[meta.split == "test"].reset_index(drop=True)
print(f"{len(meta):,} samples | {NC} classes | train {len(tr):,} val {len(va):,} test {len(te):,}")

# ---- data pipeline -------------------------------------------------------
# No memmap cache. It was the source of two separate failures (a 24.6 GB file
# that was disk-bound, then a 21 GB one that was OOM-killed) and reading PNGs
# directly costs ~90 img/s, which is fine when the GPU is the bottleneck anyway.
NORM = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

def tf_train(size):
    # Label-preserving only. NO horizontal/vertical flip: mirroring turns
    # rel_strong_positive into rel_strong_negative and ts_increasing into
    # ts_decreasing, which are separate classes here.
    return T.Compose([
        T.Resize((size, size)),
        T.RandomApply([T.ColorJitter(0.25, 0.25, 0.2, 0.03)], p=0.7),
        T.RandomApply([T.RandomResizedCrop(size, scale=(0.85, 1.0), ratio=(0.95, 1.05))], p=0.5),
        T.RandomApply([T.GaussianBlur(3, (0.1, 1.2))], p=0.2),
        T.ToTensor(), NORM, T.RandomErasing(p=0.15, scale=(0.01, 0.04))])

def tf_eval(size):
    return T.Compose([T.Resize((size, size)), T.ToTensor(), NORM])

class ChartDS(Dataset):
    def __init__(self, df, size, train=False, crop_plot=False):
        self.df = df.reset_index(drop=True)
        self.tf = tf_train(size) if train else tf_eval(size)
        self.crop = crop_plot
        self.boxes = {}
        if crop_plot:
            for sp in ("train", "val", "test"):
                with open(DATA / "splits" / f"{sp}.jsonl") as fh:
                    for line in fh:
                        r = json.loads(line)
                        b = next((x["bbox"] for x in r["render"]["grounding_boxes"]
                                  if x["label"] == "plot_area"), None)
                        if b: self.boxes[r["sample_id"]] = b
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(DATA / r.image).convert("RGB")
        if self.crop:
            b = self.boxes.get(r.sample_id)
            if b:
                x0, y0, x1, y1 = [int(round(v)) for v in b]
                if x1 - x0 > 20 and y1 - y0 > 20: img = img.crop((x0, y0, x1, y1))
        return self.tf(img), C2I[r.label], i

def loader(df, size, train, bs=48, crop=False):
    # modest workers, no pinned memory, no persistent workers: the combination
    # of persistent_workers + pin_memory leaked ~5 GB per model
    return DataLoader(ChartDS(df, size, train, crop), batch_size=bs, shuffle=train,
                      num_workers=2, pin_memory=False, prefetch_factor=2,
                      persistent_workers=False)

def build(name, n=NC):
    if name == "resnet50":
        m = M.resnet50(weights=M.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "resnet101":
        m = M.resnet101(weights=M.ResNet101_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "densenet121":
        m = M.densenet121(weights=M.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, n)
    elif name == "resnet18":
        m = M.resnet18(weights=M.ResNet18_Weights.IMAGENET1K_V1); m.fc = nn.Linear(m.fc.in_features, n)
    else: raise ValueError(name)
    return m

def macro_f1(P, Y, n=NC):
    fs = []
    for i in range(n):
        tp = ((P==i)&(Y==i)).sum(); fp = ((P==i)&(Y!=i)).sum(); fn = ((P!=i)&(Y==i)).sum()
        pr = tp/(tp+fp) if tp+fp else 0; rc = tp/(tp+fn) if tp+fn else 0
        fs.append(2*pr*rc/(pr+rc) if pr+rc else 0)
    return float(np.mean(fs))

@torch.no_grad()
def predict(model, dl):
    model.eval(); P=[]; Y=[]; IX=[]
    for x, y, ix in dl:
        with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
            o = model(x.to(DEVICE, non_blocking=True))
        P.append(o.float().argmax(1).cpu()); Y.append(y); IX.append(ix)
    return torch.cat(P).numpy(), torch.cat(Y).numpy(), torch.cat(IX).numpy()

def train_model(name, size, epochs, bs=48, crop=False, patience=4, tag=""):
    dl_tr, dl_va = loader(tr, size, True, bs, crop), loader(va, size, False, bs, crop)
    model = build(name).to(DEVICE)
    if DEVICE.type == "cuda" and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, epochs=epochs,
                                                steps_per_epoch=len(dl_tr), pct_start=0.25)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type=="cuda")
    ckpt = f"{name}{tag}_best.pt"; best = -1.0; stale = 0; hist = []
    for ep in range(1, epochs+1):
        model.train(); t0 = time.time(); tot=n=corr=0
        for bi, (x, y, _) in enumerate(dl_tr):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=DEVICE.type=="cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.detach().item()*y.size(0); corr += (out.detach().argmax(1)==y).sum().item(); n += y.size(0)
            if bi % 60 == 0:
                print(f"\r  ep{ep} [{bi}/{len(dl_tr)}] loss {tot/max(n,1):.3f} acc {corr/max(n,1):.3f}", end="", flush=True)
        P, Y, _ = predict(model, dl_va); vf1 = macro_f1(P, Y)
        hist.append(dict(epoch=ep, loss=tot/n, train_acc=corr/n, val_acc=float((P==Y).mean()), val_f1=vf1))
        print(f"\r  ep{ep}  loss {tot/n:.3f}  train {corr/n:.3f}  val_acc {(P==Y).mean():.4f}  val_F1 {vf1:.4f}  ({time.time()-t0:.0f}s)")
        if vf1 > best:
            best, stale = vf1, 0
            torch.save((model.module if isinstance(model, nn.DataParallel) else model).state_dict(), ckpt)
        else:
            stale += 1
            if stale >= patience:
                print(f"  early stop: val macro-F1 flat for {patience} epochs"); break
    clean = build(name).to(DEVICE); clean.load_state_dict(torch.load(ckpt)); clean.eval()
    del model, dl_tr, dl_va, opt, sched, scaler; gc.collect(); torch.cuda.empty_cache()
    return clean, hist, best, ckpt

def save(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))
    print("wrote", path)
print("preamble ready")

In [ ]:
# Same backbone and epoch budget as NB1: only resolution differs.
MODEL, SIZE, EPOCHS, BS = "resnet101", 384, 12, 12
print(f"=== {MODEL} @ {SIZE}px, batch {BS} ===")
t0 = time.time()
cnn, hist, best_val, _ = train_model(MODEL, SIZE, EPOCHS, bs=BS, tag="_384")
P, Y, IX = predict(cnn, loader(te, SIZE, False, bs=BS))
acc, f1 = float((P==Y).mean()), macro_f1(P, Y)
prov = te.provenance.values[IX]
print(f"\n@{SIZE}px: accuracy {acc:.4f}  macro-F1 {f1:.4f}  ({(time.time()-t0)/60:.1f} min)")
print(f"  real-only      {float((P==Y)[prov=='real'].mean()):.4f}")
print(f"  synthetic-only {float((P==Y)[prov=='synthetic'].mean()):.4f}")
print("\nCompare against NB1 @224px: the difference is the resolution effect.")
save({"model":MODEL,"size":SIZE,"epochs":EPOCHS,"batch":BS,
      "accuracy":acc,"macro_f1":f1,"best_val_f1":best_val,
      "real_only":float((P==Y)[prov=='real'].mean()),
      "synthetic_only":float((P==Y)[prov=='synthetic'].mean()),
      "history":hist}, "nb3_results.json")